# Detección de Muelas del Juicio — Semana 2: Preparación del Dataset

**Materia:** Redes Neuronales  
**Docente:** Ing. Pablo Marinozi  
**Repo:** https://github.com/JulianOliveraBalls/dentex-wisdom-teeth

---

## Contexto del problema

Se busca detectar **muelas del juicio** (terceros molares) en radiografías panorámicas
dentales y clasificar su estado:

| Clase | Descripción clínica | Señal visual |
|-------|---------------------|-------------|
| `erupted` (0) | Muela erupcionada normalmente | Diente en posición normal — similar a molares adyacentes |
| `impacted` (1) | Muela retenida/impactada | Diente en posición horizontal o diagonal — señal distintiva |

### Decisiones de diseño

| Problema | Solución |
|----------|----------|
| Pocas imágenes (~430 con muelas) | Augmentation + cuadrantes + dataset adicional ExAn-MTM |
| Desbalance erupted/impacted (~37/63%) | Augmentation estratificada en train |
| Panorámicas anchas (~2800px) | División en 4 cuadrantes solapados → muelas más grandes en el frame |
| `erupted` visualmente ambigua | Limitación documentada — requiere contexto global del arco dental |

| Sección | Contenido |
|---------|----------|
| 0 | Setup — detección automática Colab/local |
| a | Carga y estructura del dataset |
| b | Particionado estratificado 70/5/25 |
| c | Preprocesamiento |
| d | Data augmentation (6 estrategias) |
| e | DataLoaders + verificación del batch |
| f | Exportar CSVs de splits |


## Sección 0 — Setup

In [1]:
import subprocess, sys, os, gc, json, random, warnings, shutil, zipfile
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pkgs, '-q'])

try:
    import ultralytics, sklearn, huggingface_hub
except ImportError:
    pip_install('ultralytics', 'scikit-learn', 'huggingface_hub')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from huggingface_hub import hf_hub_download as hf_dl

# ── Detección automática Colab / local ────────────────────────────────────────
try:
    import google.colab; IN_COLAB = True
    from google.colab import drive; drive.mount('/drive')
    # En Colab: clonamos el repo si no existe
    REPO_ROOT = Path('/content/dentex-wisdom-teeth')
    if not REPO_ROOT.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/JulianOliveraBalls/dentex-wisdom-teeth.git',
                        str(REPO_ROOT)], check=True)
    DRIVE_DIR = Path('/drive/MyDrive/dentex_runs')
except ImportError:
    IN_COLAB  = False
    # Local: la raíz del repo está 1 nivel arriba del notebook (dev/)
    REPO_ROOT = Path('__file__').resolve().parent.parent if '__file__' in dir() else Path.cwd().parent
    DRIVE_DIR = REPO_ROOT / 'dev'   # pesos en dev/ localmente

# ── Paths derivados del repo ──────────────────────────────────────────────────
DATA_DIR    = REPO_ROOT / 'data'
RAW_DIR     = DATA_DIR / 'raw'
DENTEX_DIR  = RAW_DIR / 'dentex_raw'
EXAN_DIR    = RAW_DIR / 'exan_mtm'
PROCESSED   = DATA_DIR / 'processed'
YOLO_DIR    = PROCESSED / 'yolo_dataset'
YOLO_MERGED = PROCESSED / 'yolo_merged'
OUTPUTS_DIR = DATA_DIR / 'outputs'
SRC_DIR     = REPO_ROOT / 'src'
for d in [OUTPUTS_DIR, SRC_DIR, DENTEX_DIR, EXAN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Agregar src/ al path para importar augmentations_config ───────────────────
sys.path.insert(0, str(SRC_DIR))

SEED = 42
BATCH_SIZE = 8
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def is_real_file(p):
    try: return Path(p).resolve().exists() and Path(p).stat().st_size > 0
    except: return False

def log(msg, level='INFO'):
    icons = {'INFO':'[INFO]','OK':'[OK]  ','WARN':'[WARN]','ERR':'[ERR] ','DATA':'[DATA]'}
    print(f'{icons.get(level,"[INFO]")} {msg}')

def split_existe(d):
    return Path(d).exists() and all(
        len(list((Path(d)/'images'/s).glob('*.*'))) > 0
        for s in ['train','val','test'])

log(f'REPO_ROOT: {REPO_ROOT}', 'DATA')
log(f'IN_COLAB:  {IN_COLAB}   device: {device}   SEED: {SEED}', 'OK')


Mounted at /drive
[DATA] REPO_ROOT: /content/dentex-wisdom-teeth
[OK]   IN_COLAB:  True   device: cpu   SEED: 42


## a) Carga y organización del dataset

### Fuentes

| Dataset | Fuente | Imágenes | Formato | Licencia |
|---------|--------|----------|---------|----------|
| **DENTEX MICCAI 2023** | [HuggingFace](https://huggingface.co/datasets/ibrahimhamamci/DENTEX) | 705 anotadas | COCO JSON | CC-BY |
| **ExAn-MTM** | [Kaggle](https://www.kaggle.com/datasets/ikyd26/expert-annotated-mandibular-third-molar-dataset) + [Figshare](https://doi.org/10.6084/m9.figshare.21621705) | 973 panorámicas | YOLO TXT | CC-BY |

Las dos fuentes usan el mismo mapeo de clases: `0=erupted`, `1=impacted`.

### Por qué no ImageFolder

`ImageFolder` asume una imagen = una clase (clasificación). Nuestro problema es
**detección**: cada imagen puede tener múltiples boxes con distintas clases.
Se implementa `DentexDataset(torch.utils.data.Dataset)` personalizado.

### Descarga automática

El script `data/download.py` descarga todo desde cero. Este notebook llama
las mismas funciones internamente con verificación de caché.


In [ ]:
import requests as _req

# ── Descarga DENTEX ───────────────────────────────────────────────────────────
json_files = [f for f in DENTEX_DIR.rglob('*.json') if is_real_file(f)] if DENTEX_DIR.exists() else []
if not any('disease' in f.name.lower() for f in json_files):
    log('Descargando DENTEX desde HuggingFace...', 'WARN')
    for fname, cname in [('DENTEX/training_data.zip','training_data'),
                          ('DENTEX/validation_data.zip','validation_data')]:
        check = DENTEX_DIR / 'DENTEX' / cname
        if check.exists() and len(list(check.rglob('*.png'))) > 0: continue
        zp = hf_dl(repo_id='ibrahimhamamci/DENTEX', repo_type='dataset',
                   filename=fname, local_dir=str(DENTEX_DIR))
        with zipfile.ZipFile(zp, 'r') as z: z.extractall(str(DENTEX_DIR/'DENTEX'))
        os.remove(zp); log(f'{fname} extraído', 'OK')
    json_files = [f for f in DENTEX_DIR.rglob('*.json') if is_real_file(f)]
else:
    log('DENTEX ya descargado', 'OK')

TRAIN_JSON = next((j for j in json_files if 'disease' in j.name.lower()), None)
if not TRAIN_JSON: raise FileNotFoundError('JSON DENTEX no encontrado')
IMG_DIR_C = next((c for c in [TRAIN_JSON.parent/'xrays', TRAIN_JSON.parent.parent/'xrays']
                  if c.exists()), TRAIN_JSON.parent/'xrays')
with open(TRAIN_JSON) as f: data_c = json.load(f)
id2img = {img['id']: img for img in data_c['images']}
wisdom = defaultdict(list)
for ann in data_c['annotations']:
    if ann.get('category_id_2') == 7: wisdom[ann['image_id']].append(ann)

def classify_image(img_id):
    return 'impacted' if any(a.get('category_id_3')==0 for a in wisdom[img_id]) else 'erupted'

counts = Counter(classify_image(i) for i in wisdom)
total  = sum(counts.values())
log(f'Imágenes con muela del juicio: {total}', 'DATA')
log(f'  erupted:  {counts["erupted"]}  ({counts["erupted"]/total*100:.1f}%)', 'DATA')
log(f'  impacted: {counts["impacted"]}  ({counts["impacted"]/total*100:.1f}%)', 'DATA')


[WARN] Descargando DENTEX desde HuggingFace...


DENTEX/training_data.zip:   0%|          | 0.00/10.9G [00:00<?, ?B/s]

In [ ]:
# ── Descarga ExAn-MTM ─────────────────────────────────────────────────────────
KAGGLE_TOKEN = 'KGAT_9541105c6a2fec68be43345f171bc2c9'
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_TOKEN
token_path = Path.home() / '.kaggle' / 'access_token'
token_path.parent.mkdir(exist_ok=True)
token_path.write_text(KAGGLE_TOKEN); token_path.chmod(0o600)
try: import kaggle as _k
except ImportError: pip_install('kaggle')

EXAN_BASE      = EXAN_DIR / 'ExAn-MTM dataset'
EXAN_TRAIN_IMG = EXAN_BASE / 'train' / 'images'
EXAN_TRAIN_LBL = EXAN_BASE / 'train' / 'labels'
EXAN_VAL_IMG   = EXAN_BASE / 'valid' / 'images'
EXAN_VAL_LBL   = EXAN_BASE / 'valid' / 'labels'

n_tr = len(list(EXAN_TRAIN_IMG.glob('*.jpg'))) if EXAN_TRAIN_IMG.exists() else 0
if n_tr >= 875:
    log(f'ExAn-MTM ya descargado: {n_tr} imgs', 'OK')
else:
    log('Descargando labels ExAn-MTM desde Kaggle...', 'WARN')
    r = subprocess.run(
        ['python','-m','kaggle','datasets','download','-d',
         'ikyd26/expert-annotated-mandibular-third-molar-dataset',
         '--path', str(EXAN_DIR), '--unzip'],
        capture_output=True, text=True,
        env={**os.environ,'KAGGLE_API_TOKEN':KAGGLE_TOKEN})
    if r.returncode != 0: log(f'Error: {r.stderr[:200]}','ERR')
    else: log('Labels OK','OK')

    ZIP_PATH = EXAN_DIR / 'dental_dataset.zip'
    EXAN_TRAIN_IMG.mkdir(parents=True,exist_ok=True)
    EXAN_VAL_IMG.mkdir(parents=True,exist_ok=True)
    if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size < 1e9:
        log('Descargando imágenes UESB desde Figshare (1.6 GB)...','WARN')
        with _req.get('https://ndownloader.figshare.com/files/38322366',stream=True) as resp:
            resp.raise_for_status()
            total_b = int(resp.headers.get('content-length',0))
            downloaded = 0
            with open(ZIP_PATH,'wb') as f:
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    f.write(chunk); downloaded += len(chunk)
                    if downloaded % (200*1024*1024)==0:
                        log(f'  {downloaded/1e6:.0f}/{total_b/1e6:.0f} MB','DATA')
        log('ZIP descargado','OK')
    with zipfile.ZipFile(ZIP_PATH,'r') as z:
        all_names = z.namelist()
        imgs_tr = {Path(n).stem:n for n in all_names
                   if 'Dataset and code/train/images' in n and n.lower().endswith(('.jpg','.png'))}
        imgs_val= {Path(n).stem:n for n in all_names
                   if 'Dataset and code/test/images'  in n and n.lower().endswith(('.jpg','.png'))}
        for stem in [p.stem for p in EXAN_TRAIN_LBL.glob('*.txt')]:
            dst=EXAN_TRAIN_IMG/f'{stem}.jpg'
            if not dst.exists() and stem in imgs_tr: dst.write_bytes(z.read(imgs_tr[stem]))
        for stem in [p.stem for p in EXAN_VAL_LBL.glob('*.txt')]:
            src=imgs_tr.get(stem) or imgs_val.get(stem)
            dst=EXAN_VAL_IMG/f'{stem}.jpg'
            if not dst.exists() and src: dst.write_bytes(z.read(src))
    log(f'ExAn-MTM: {len(list(EXAN_TRAIN_IMG.glob("*.jpg")))} train, '
        f'{len(list(EXAN_VAL_IMG.glob("*.jpg")))} val','OK')


## b) Particionado

| Decisión | Valor | Justificación |
|----------|-------|---------------|
| Proporciones | **70/5/25** | Train máximo, test representativo |
| Estratificado | `stratify=labels` | Mantiene ratio erupted/impacted en los 3 splits |
| Semilla | `seed=42` | Reproducibilidad garantizada |
| Test set | Solo DENTEX | Evaluación honesta — ExAn-MTM nunca aparece en test |

### Expansión con cuadrantes (solo train)

```
panorámica 2800×1300px  →  4 cuadrantes con 10% solapamiento
308 imgs train  →  ~1744 imgs train   (308 × 4 cuadrantes × 2 con flip)
```

Las bounding boxes se reproyectan al sistema de coordenadas de cada cuadrante.
Solo se incluyen cuadrantes con al menos 1 box con >=50% de área visible.


In [ ]:
def generar_split_yolo(yolo_out, img_ids, labels, wisdom_dict, id2img_dict, img_dir):
    ids_tr,ids_tmp,_,y_tmp = train_test_split(img_ids,labels,test_size=0.30,stratify=labels,random_state=SEED)
    ids_val,ids_test,_,_  = train_test_split(ids_tmp,y_tmp,test_size=0.833,stratify=y_tmp,random_state=SEED)
    for split,ids in {'train':ids_tr,'val':ids_val,'test':ids_test}.items():
        (yolo_out/'images'/split).mkdir(parents=True,exist_ok=True)
        (yolo_out/'labels'/split).mkdir(parents=True,exist_ok=True)
        for img_id in ids:
            info=id2img_dict[img_id]; src=img_dir/Path(info['file_name']).name
            if not is_real_file(src): continue
            pil=Image.open(src); W,H=pil.size
            dst=yolo_out/'images'/split/src.name
            if not dst.exists(): os.symlink(src.resolve(),dst)
            lines=[]
            for ann in wisdom_dict[img_id]:
                x,y,bw,bh=ann['bbox']
                xc=min(max((x+bw/2)/W,0),1); yc=min(max((y+bh/2)/H,0),1)
                wn=min(max(bw/W,0),1); hn=min(max(bh/H,0),1)
                cls=1 if ann.get('category_id_3')==0 else 0
                lines.append(f'{cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            (yolo_out/'labels'/split/(src.stem+'.txt')).write_text('\n'.join(lines))
    (yolo_out/'dataset.yaml').write_text(
        f'path: {yolo_out}\ntrain: images/train\nval:   images/val\n'
        f'test:  images/test\nnc: 2\nnames:\n  0: erupted\n  1: impacted\n')

def expandir_cuadrantes(src,dst,overlap=0.1):
    for split in ['val','test']:
        for sub in ['images','labels']:
            d=dst/sub/split; d.mkdir(parents=True,exist_ok=True)
            for f in (src/sub/split).glob('*.*'):
                lnk=d/f.name
                if not lnk.exists(): os.symlink(f.resolve(),lnk)
    (dst/'images'/'train').mkdir(parents=True,exist_ok=True)
    (dst/'labels'/'train').mkdir(parents=True,exist_ok=True)
    n_o=n_q=0
    for img_p in sorted((src/'images'/'train').glob('*.png')):
        lbl_p=src/'labels'/'train'/(img_p.stem+'.txt')
        if not lbl_p.exists(): continue
        pil=Image.open(img_p).convert('RGB'); W,H=pil.size
        d_o=dst/'images'/'train'/img_p.name
        if not d_o.exists(): os.symlink(img_p.resolve(),d_o)
        shutil.copy2(lbl_p,dst/'labels'/'train'/lbl_p.name); n_o+=1
        boxes=[(int(p[0]),*map(float,p[1:])) for ln in lbl_p.read_text().strip().split('\n') if ln for p in [ln.split()]]
        ox=int(W*overlap/2); oy=int(H*overlap/2)
        quads=[('tl',0,0,W//2+ox,H//2+oy),('tr',W//2-ox,0,W,H//2+oy),
               ('bl',0,H//2-oy,W//2+ox,H),('br',W//2-ox,H//2-oy,W,H)]
        for qn,x1,y1,x2,y2 in quads:
            qW,qH=x2-x1,y2-y1; crop=pil.crop((x1,y1,x2,y2)); qlines=[]
            for cls,xc,yc,bw,bh in boxes:
                ax1=xc*W-bw*W/2; ay1=yc*H-bh*H/2; ax2=xc*W+bw*W/2; ay2=yc*H+bh*H/2
                ix1=max(ax1,x1)-x1; iy1=max(ay1,y1)-y1; ix2=min(ax2,x2)-x1; iy2=min(ay2,y2)-y1
                if ix2<=ix1 or iy2<=iy1: continue
                if (ix2-ix1)*(iy2-iy1)/(bw*W*bh*H)<0.5: continue
                nxc=min(max((ix1+ix2)/2/qW,0),1); nyc=min(max((iy1+iy2)/2/qH,0),1)
                nw=min(max((ix2-ix1)/qW,0),1); nh=min(max((iy2-iy1)/qH,0),1)
                qlines.append(f'{cls} {nxc:.6f} {nyc:.6f} {nw:.6f} {nh:.6f}')
            if not qlines: continue
            st=f'{img_p.stem}_{qn}'; crop.save(str(dst/'images'/'train'/(st+'.png')))
            (dst/'labels'/'train'/(st+'.txt')).write_text('\n'.join(qlines)); n_q+=1
            fl=crop.transpose(Image.FLIP_LEFT_RIGHT)
            flines=[f'{l.split()[0]} {1-float(l.split()[1]):.6f} {" ".join(l.split()[2:])}' for l in qlines]
            sf=f'{img_p.stem}_{qn}_flip'; fl.save(str(dst/'images'/'train'/(sf+'.png')))
            (dst/'labels'/'train'/(sf+'.txt')).write_text('\n'.join(flines)); n_q+=1
    (dst/'dataset.yaml').write_text(
        f'path: {dst}\ntrain: images/train\nval:   images/val\n'
        f'test:  images/test\nnc: 2\nnames:\n  0: erupted\n  1: impacted\n')
    return n_o,n_q

def construir_merged(src_dentex,exan_ti,exan_tl,exan_vi,exan_vl,dst):
    for split in ['train','val','test']:
        (dst/'images'/split).mkdir(parents=True,exist_ok=True)
        (dst/'labels'/split).mkdir(parents=True,exist_ok=True)
    n=defaultdict(int)
    for split in ['train','val','test']:
        for img_p in sorted(list((src_dentex/'images'/split).glob('*.png'))+
                             list((src_dentex/'images'/split).glob('*.jpg'))):
            lbl_p=src_dentex/'labels'/split/(img_p.stem+'.txt')
            if not lbl_p.exists(): continue
            di=dst/'images'/split/f'dentex_{img_p.name}'; dl=dst/'labels'/split/f'dentex_{img_p.stem}.txt'
            if not di.exists():
                real=Path(os.path.realpath(str(img_p)))
                if real.exists() and real!=di: os.symlink(real,di)
                else: shutil.copy2(str(img_p),str(di))
            if not dl.exists(): shutil.copy2(str(lbl_p),str(dl))
            n[f'dentex_{split}']+=1
    for imgs_src,lbl_src,split_dst in [(exan_ti,exan_tl,'train'),(exan_vi,exan_vl,'val')]:
        if not imgs_src or not imgs_src.exists(): continue
        for img_p in sorted(list(imgs_src.glob('*.jpg'))+list(imgs_src.glob('*.png'))):
            lbl_p=lbl_src/(img_p.stem+'.txt') if lbl_src else None
            if not lbl_p or not lbl_p.exists(): continue
            di=dst/'images'/split_dst/f'exan_{img_p.name}'; dl=dst/'labels'/split_dst/f'exan_{img_p.stem}.txt'
            if not di.exists(): os.symlink(img_p.resolve(),di)
            if not dl.exists(): shutil.copy2(str(lbl_p),str(dl))
            n[f'exan_{split_dst}']+=1
    (dst/'dataset.yaml').write_text(
        f'path: {dst}\ntrain: images/train\nval:   images/val\n'
        f'test:  images/test\nnc: 2\nnames:\n  0: erupted\n  1: impacted\n')
    return dict(n)

# Generar todos los datasets
YOLO_QUAD = PROCESSED / 'yolo_quad_flip'
if split_existe(YOLO_DIR): log(f'yolo_dataset: ya existe','OK')
else:
    img_ids=list(wisdom.keys()); labels=[classify_image(i) for i in img_ids]
    generar_split_yolo(YOLO_DIR,img_ids,labels,wisdom,id2img,IMG_DIR_C); log('yolo_dataset generado','OK')
if split_existe(YOLO_QUAD): log(f'yolo_quad_flip: ya existe','OK')
else:
    n_o,n_q=expandir_cuadrantes(YOLO_DIR,YOLO_QUAD); log(f'yolo_quad_flip: {n_o+n_q} imgs train','OK')
if split_existe(YOLO_MERGED): log(f'yolo_merged: ya existe','OK')
else:
    stats=construir_merged(YOLO_QUAD,EXAN_TRAIN_IMG,EXAN_TRAIN_LBL,EXAN_VAL_IMG,EXAN_VAL_LBL,YOLO_MERGED)
    for k,v in stats.items(): log(f'  {k}: {v}','DATA')

log('Distribución por split (seed=42, estratificado):','DATA')
for split in ['train','val','test']:
    img_d=YOLO_MERGED/'images'/split; lbl_d=YOLO_MERGED/'labels'/split
    n=len(list(img_d.glob('*.*'))); c0=c1=0
    for lbl in lbl_d.glob('*.txt'):
        for line in lbl.read_text().strip().split('\n'):
            if line: (c0 if int(line.split()[0])==0 else None) or None
            if line: c0+=int(line.split()[0])==0; c1+=int(line.split()[0])==1
    log(f'  {split:5s}: {n:5d} imgs | erupted={c0} ({c0/max(c0+c1,1)*100:.0f}%) impacted={c1} ({c1/max(c0+c1,1)*100:.0f}%)','DATA')


## c) Preprocesamiento

| Paso | Valor | Justificación |
|------|-------|---------------|
| `Resize(640,640)` | 640×640 px | Resolución nativa de YOLOv8 |
| `ToTensor()` | float32 [0,1] | Convierte PIL uint8 [0,255] |
| `Normalize(ImageNet)` | mean/std de COCO | El backbone fue preentrenado con estas estadísticas |

Las panorámicas son grises (escala de grises). Se cargan como RGB: los 3 canales son idénticos.

**¿Por qué 640 y no 224?** YOLOv8 necesita resolver objetos pequeños — las muelas
ocupan ~30-50 px en panorámicas completas. Con 224px serían ~8 px, indetectables.


In [ ]:
# Importar augmentations desde src/ (ya está en sys.path)
try:
    from augmentations_config import AUGMENTATIONS, BASE_TRANSFORM, MEAN, STD
    log('augmentations_config importado desde src/', 'OK')
except ImportError:
    log('src/augmentations_config.py no encontrado — definiendo inline', 'WARN')
    MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
    BASE_TRANSFORM = T.Compose([
        T.Resize((640,640)), T.ToTensor(), T.Normalize(mean=MEAN,std=STD)])
    AUGMENTATIONS = {'A_baseline': BASE_TRANSFORM}

MEAN_T = torch.tensor(MEAN).view(3,1,1)
STD_T  = torch.tensor(STD).view(3,1,1)

class DentexDataset(Dataset):
    """Dataset YOLO de radiografías panorámicas.
    Args:
        split:    'train', 'val' o 'test'
        yolo_dir: Path al directorio con images/ y labels/
        transform: torchvision transform a aplicar
    """
    CLASS_NAMES  = {0:'erupted', 1:'impacted'}
    CLASS_COLORS = {0:'#44CC44', 1:'#FF4444'}

    def __init__(self, split, yolo_dir, transform=None):
        self.split=split; self.yolo_dir=Path(yolo_dir); self.transform=transform
        self.img_dir=self.yolo_dir/'images'/split; self.lbl_dir=self.yolo_dir/'labels'/split
        self.samples=[(p,self.lbl_dir/(p.stem+'.txt'))
            for p in sorted(list(self.img_dir.glob('*.png'))+list(self.img_dir.glob('*.jpg')))
            if (self.lbl_dir/(p.stem+'.txt')).exists()]
        self.class_counts=Counter()
        for _,lp in self.samples:
            for line in lp.read_text().strip().split('\n'):
                if line: self.class_counts[int(line.split()[0])]+=1
        log(f'DentexDataset [{split}]: {len(self.samples)} imgs | '
            f'erupted={self.class_counts[0]} impacted={self.class_counts[1]}','OK')

    def __len__(self): return len(self.samples)

    def __getitem__(self,idx):
        img_path,lbl_path=self.samples[idx]
        image=Image.open(img_path).convert('RGB')
        boxes,labels=[],[]
        for line in lbl_path.read_text().strip().split('\n'):
            if line: p=line.split(); labels.append(int(p[0])); boxes.append(list(map(float,p[1:])))
        if self.transform: image=self.transform(image)
        return image, {'boxes':torch.tensor(boxes,dtype=torch.float32) if boxes else torch.zeros((0,4)),
                       'labels':torch.tensor(labels,dtype=torch.long) if labels else torch.zeros(0,dtype=torch.long),
                       'image_id':img_path.stem}

    def get_image_raw(self,idx): return Image.open(self.samples[idx][0]).convert('RGB')

def collate_fn(batch):
    """Agrupa batch con distinto número de boxes por imagen."""
    return torch.stack([b[0] for b in batch]), [b[1] for b in batch]

# Verificación
ds_tmp=DentexDataset('train',YOLO_MERGED,transform=BASE_TRANSFORM)
img_raw=ds_tmp.get_image_raw(0); img_t,_=ds_tmp[0]
log(f'PIL: {img_raw.size} modo={img_raw.mode}  →  Tensor: {tuple(img_t.shape)} [{img_t.min():.2f},{img_t.max():.2f}]','DATA')
arr=np.array(img_raw)
log(f'Grises confirmado (R==G=={np.array_equal(arr[:,:,0],arr[:,:,1])}, R==B=={np.array_equal(arr[:,:,0],arr[:,:,2])})','DATA')


## d) Data Augmentation

Las 6 estrategias están definidas en `src/augmentations_config.py`.
Se aplican **únicamente al train**. Val y test siempre usan `BASE_TRANSFORM`.

| ID | Nombre | Transformaciones |
|----|--------|------------------|
| A | `baseline` | Solo resize+normalize — referencia limpia |
| B | `flips` | + HorizontalFlip(p=0.5) — simetría bilateral del arco dental |
| C | `color` | + ColorJitter + RandomAffine(5°) — variabilidad entre equipos |
| D | `geometric` | + RandomAffine(8°,scale) + RandomErasing — artefactos metálicos |
| E | `full` | B + C + D combinados |
| F | `mixaug` | E + GaussianBlur — variabilidad de resolución |

**Excluidas:** VerticalFlip (anatómicamente incorrecto), mosaic/mixup (mezcla anatomías distintas).


In [ ]:
# Visualización del efecto de cada augmentation
ERASING_VIS = T.RandomErasing(p=0.9, scale=(0.05,0.15), value='random')  # forzado para visualizar

def get_transform_vis(aug_name):
    t = AUGMENTATIONS[aug_name]
    if aug_name in ('D_geometric','E_full','F_mixaug'):
        steps=[s for s in t.transforms if not isinstance(s,T.RandomErasing)]
        return T.Compose(steps+[ERASING_VIS])
    return t

sample_raws=[ds_tmp.get_image_raw(i*len(ds_tmp)//4) for i in range(4)]
N_AUG=len(AUGMENTATIONS)
fig,axes=plt.subplots(N_AUG,5,figsize=(20,3*N_AUG))
for row,(aug_name,_) in enumerate(AUGMENTATIONS.items()):
    aug_vis=get_transform_vis(aug_name)
    axes[row][0].imshow(np.array(sample_raws[0].resize((320,320))))
    axes[row][0].set_ylabel(aug_name,fontsize=8,rotation=0,labelpad=100,va='center')
    if row==0: axes[row][0].set_title('Original',fontsize=8)
    axes[row][0].axis('off')
    for col,raw in enumerate(sample_raws):
        aug_t=aug_vis(raw.convert('RGB').resize((640,640)))
        if isinstance(aug_t,torch.Tensor):
            vis=(aug_t*STD_T+MEAN_T).clamp(0,1).permute(1,2,0).numpy()
        else: vis=np.array(aug_t)
        axes[row][col+1].imshow(vis)
        if row==0: axes[row][col+1].set_title(f'Img {col+1}',fontsize=8)
        axes[row][col+1].axis('off')
plt.suptitle('Augmentations (A-F) — D/E/F: RandomErasing forzado p=0.9 para visualización\n'
             'En producción: p=0.10/0.15  |  Solo se aplican en TRAIN')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR/'augmentations_comparacion.png'),dpi=80,bbox_inches='tight')
plt.show()


## e) DataLoaders y verificación del batch

| Parámetro | Train | Val/Test | Justificación |
|-----------|-------|----------|---------------|
| `batch_size` | 8 | 8 | Limitado por VRAM T4 con imágenes 640×640 |
| `shuffle` | True | False | Train: variabilidad. Val/Test: deterministas |
| `collate_fn` | personalizado | personalizado | Distinto número de boxes por imagen |
| `num_workers` | 2 | 2 | Carga paralela sin saturar RAM de Colab |


In [ ]:
# DataLoaders — un loader por augmentation en train
dl_train_augs={}
for aug_name,aug_transform in AUGMENTATIONS.items():
    ds=DentexDataset('train',YOLO_MERGED,transform=aug_transform)
    dl=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,
                  collate_fn=collate_fn,pin_memory=torch.cuda.is_available())
    dl_train_augs[aug_name]=dl
    log(f'  dl_train[{aug_name}]: {len(ds)} imgs, {len(dl)} batches','DATA')

# Val y test SIEMPRE sin augmentation
ds_val  = DentexDataset('val',  YOLO_MERGED, transform=BASE_TRANSFORM)
ds_test = DentexDataset('test', YOLO_MERGED, transform=BASE_TRANSFORM)
dl_val  = DataLoader(ds_val,  batch_size=BATCH_SIZE,shuffle=False,num_workers=2,collate_fn=collate_fn)
dl_test = DataLoader(ds_test, batch_size=BATCH_SIZE,shuffle=False,num_workers=2,collate_fn=collate_fn)
log(f'dl_val:  {len(ds_val)} imgs  |  dl_test: {len(ds_test)} imgs','OK')


In [ ]:
# Verificación del batch
batch_imgs,batch_targets=next(iter(dl_train_augs['E_full']))
log(f'Shape:  {tuple(batch_imgs.shape)}  (batch × canales × alto × ancho)','DATA')
log(f'Dtype:  {batch_imgs.dtype}','DATA')
log(f'Rango:  [{batch_imgs.min():.3f}, {batch_imgs.max():.3f}]  (normalizado ImageNet)','DATA')
log(f'Boxes:  {[len(t["boxes"]) for t in batch_targets]} por imagen','DATA')
assert not torch.isnan(batch_imgs).any() and not torch.isinf(batch_imgs).any()
log('Sin NaN ni Inf','OK')

n_show=min(8,len(batch_targets))
fig,axes=plt.subplots(2,4,figsize=(20,8))
for idx,ax in enumerate(axes.flatten()):
    if idx>=n_show: ax.axis('off'); continue
    img_t=batch_imgs[idx]; target=batch_targets[idx]
    vis=(img_t*STD_T+MEAN_T).clamp(0,1).permute(1,2,0).numpy()
    ax.imshow(vis)
    for box,lbl in zip(target['boxes'],target['labels']):
        xc,yc,bw,bh=box.tolist(); x1,y1=(xc-bw/2)*640,(yc-bh/2)*640
        color='#FF4444' if lbl==1 else '#44CC44'
        ax.add_patch(patches.Rectangle((x1,y1),bw*640,bh*640,linewidth=2,edgecolor=color,facecolor='none'))
        ax.text(x1+2,y1-8,'imp' if lbl==1 else 'eru',color='white',fontsize=6,fontweight='bold',
                bbox=dict(facecolor=color,alpha=0.8,pad=1,edgecolor='none'))
    fuente='DENTEX' if target['image_id'].startswith('dentex_') else 'ExAn-MTM'
    ax.set_title(f'[{fuente}] {len(target["boxes"])} boxes',fontsize=7); ax.axis('off')
plt.suptitle(f'Batch verificado (E_full) — Shape: {tuple(batch_imgs.shape)}  Verde=erupted  Rojo=impacted')
plt.tight_layout()
plt.savefig(str(OUTPUTS_DIR/'batch_verificacion.png'),dpi=100,bbox_inches='tight')
plt.show()


## f) Exportar CSVs de splits

Se exportan a `data/` del repo para versionarlos en Git.
Garantizan que cualquier clon del repo use exactamente el mismo particionado.


In [ ]:
for split in ['train','val','test']:
    rows=[]
    img_dir=YOLO_MERGED/'images'/split; lbl_dir=YOLO_MERGED/'labels'/split
    for img_path in sorted(list(img_dir.glob('*.png'))+list(img_dir.glob('*.jpg'))):
        lbl_path=lbl_dir/(img_path.stem+'.txt')
        if not lbl_path.exists(): continue
        classes=[int(l.split()[0]) for l in lbl_path.read_text().strip().split('\n') if l]
        rows.append({'filename':img_path.name,'split':split,
                     'origen':'dentex' if img_path.name.startswith('dentex_') else 'exan',
                     'img_class':'impacted' if 1 in classes else 'erupted',
                     'n_boxes':len(classes),'n_erupted':classes.count(0),'n_impacted':classes.count(1)})
    df=pd.DataFrame(rows)
    csv_path=DATA_DIR/f'{split}.csv'
    df.to_csv(csv_path,index=False)
    eru=(df.img_class=='erupted').sum(); imp=(df.img_class=='impacted').sum()
    log(f'CSV {split}: {len(df)} filas | erupted={eru} ({eru/len(df)*100:.0f}%) impacted={imp} ({imp/len(df)*100:.0f}%)','OK')

log('CSVs guardados en data/ — listos para versionar en Git','OK')
